# MVCLTrainingSleepEEG on Sleep-EDF

This example shows how to use `MVCLTrainingSleepEEG` from `pyhealth.tasks`.

The workflow mirrors the task pattern used in PyHealth examples:
1. Load `SleepEDFDataset`.
2. Run the task on one patient for a quick sanity check.
3. Optionally run `set_task()` for the full dataset.
4. Run the pretraining step on the subset of SleepEDF data.
5. Save the model state.

In [1]:
!uv pip install ipywidgets

import os

from pyhealth.datasets import SleepEDFDataset
from pyhealth.tasks import MVCLTrainingSleepEEG

# Update this absolute path to your local Sleep-EDF root.
DATA_ROOT = "C:\\Users\\shart\\workspace\\CS-598\\PyHealth\\sleepedf"
assert os.path.exists(DATA_ROOT), f"Sleep-EDF root path {DATA_ROOT} does not exist. Please update the path to your local Sleep-EDF root."
assert os.path.isabs(DATA_ROOT), f"Sleep-EDF root path {DATA_ROOT} is not an absolute path. Please update to an absolute path."

Using Python 3.13.7 environment at: C:\Users\shart\workspace\CS-598\PyHealth\.venv
Checked 1 package in 182ms


In [2]:
dataset = SleepEDFDataset(root=DATA_ROOT, subset="cassette", dev=True)
dataset.stats()
print(f"Number of patients: {len(dataset.unique_patient_ids)}")

No config path provided, using default config
Initializing sleepedf dataset from C:\Users\shart\workspace\CS-598\PyHealth\sleepedf (dev mode: True)
No cache_dir provided. Using default cache dir: C:\Users\shart\AppData\Local\pyhealth\pyhealth\Cache\c8f0e13c-fb2e-5216-8969-8e6afcc7338c
Found cached event dataframe: C:\Users\shart\AppData\Local\pyhealth\pyhealth\Cache\c8f0e13c-fb2e-5216-8969-8e6afcc7338c\global_event_df.parquet
Dataset: sleepedf
Dev mode: True
Number of patients: 78
Number of events: 153
Found 78 unique patient IDs
Number of patients: 78


In [3]:
"""
This dataset contains 153 whole-night sleep electroencephalography
(EEG) recordings collected from 82 healthy subjects. Each recording is sampled at 100 Hz using a 1-lead 
EEG signal. The EEG signals are segmented into non-overlapping windows of size 200, each forming
one sample. Each sample is labeled with one of five sleep stages: Wake (W), Non-rapid Eye Movement
(N1, N2, N3), and Rapid Eye Movement (REM). This segmentation results in 371,055 samples
"""

'\nThis dataset contains 153 whole-night sleep electroencephalography\n(EEG) recordings collected from 82 healthy subjects. Each recording is sampled at 100 Hz using a 1-lead \nEEG signal. The EEG signals are segmented into non-overlapping windows of size 200, each forming\none sample. Each sample is labeled with one of five sleep stages: Wake (W), Non-rapid Eye Movement\n(N1, N2, N3), and Rapid Eye Movement (REM). This segmentation results in 371,055 samples\n'

In [4]:
# Quick sanity check on one patient.
patient_id = dataset.unique_patient_ids[0]
patient = dataset.get_patient(patient_id)

task = MVCLTrainingSleepEEG(
    window_size=200, ## Create None overlapping window of 200 lenth 
    crop_length=178, ## take first 178 data points of the window to match that of Epilepsy data 
    eeg_channel="EEG Fpz-Cz",
    root_path=DATA_ROOT, ## Pass the root path to the task so it can load the data correctly.
)
samples = task(patient)

print(f"patient_id: {patient_id}")
print(f"sample count: {len(samples)}")
print(f"sample keys: {list(samples[0].keys())}")
print(f"signal shape: {samples[0]['signal'].shape}")
print(f"xt shape: {samples[0]['xt'].shape}")
print(f"xd shape: {samples[0]['xd'].shape}")
print(f"xf shape: {samples[0]['xf'].shape}")
print(f"label: {samples[0]['label']}")

Used Annotations descriptions: [np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage 4'), np.str_('Sleep stage R'), np.str_('Sleep stage W')]
Used Annotations descriptions: [np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage R'), np.str_('Sleep stage W')]
patient_id: 25
sample count: 81375
sample keys: ['patient_id', 'night', 'patient_age', 'patient_sex', 'epoch_index', 'window_in_epoch', 'signal', 'xt', 'xd', 'xf', 'label']
signal shape: (1, 178)
xt shape: torch.Size([178, 1])
xd shape: torch.Size([178, 1])
xf shape: torch.Size([178, 1])
label: 0


In [5]:
# Full pipeline (can take a while and uses disk cache).
sample_dataset = dataset.set_task(task, num_workers=16)
print(f"Total task samples: {len(sample_dataset)}")
print(f"Input schema: {sample_dataset.input_schema}")
print(f"Output schema: {sample_dataset.output_schema}")

Setting task MVCLTrainingSleepEEG for sleepedf base dataset...
Task cache paths: task_df=C:\Users\shart\AppData\Local\pyhealth\pyhealth\Cache\c8f0e13c-fb2e-5216-8969-8e6afcc7338c\tasks\MVCLTrainingSleepEEG_2060b134-eac9-5c04-b238-25b00edd7ba5\task_df.ld, samples=C:\Users\shart\AppData\Local\pyhealth\pyhealth\Cache\c8f0e13c-fb2e-5216-8969-8e6afcc7338c\tasks\MVCLTrainingSleepEEG_2060b134-eac9-5c04-b238-25b00edd7ba5\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Found cached task dataframe at C:\Users\shart\AppData\Local\pyhealth\pyhealth\Cache\c8f0e13c-fb2e-5216-8969-8e6afcc7338c\tasks\MVCLTrainingSleepEEG_2060b134-eac9-5c04-b238-25b00edd7ba5\task_df.ld, skipping task transformation.
Fitting processors on the dataset...
Label label vocab: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}
Processing samples and saving to C:\Users\shart\AppData\Local\pyhealth\pyhealth\Cache\c8f0e13c-fb2e-5216-8969-8e6afcc7338c\tasks\MVCLTrainingSleepEEG_2060b134-eac9-5c04-b238-25b00edd7ba5\samples_cdbbc602-34e2-5a41-8643-4c

  0%|          | 0/6224415 [00:00<?, ?it/s]

Rank 0 inferred the following `['str', 'str', 'str', 'str', 'int', 'int', 'numpy', 'tensor', 'tensor', 'tensor', 'tensor']` data format.


  0%|          | 21120/6224415 [00:03<17:51, 5788.81it/s]c:\Users\shart\workspace\CS-598\PyHealth\.venv\Lib\site-packages\litdata\streaming\writer.py:284: UserWarning: An item was larger than the target chunk size (64.0 MB). The current chunk will be 64.1 MB in size.
  warnings.warn(
100%|██████████| 6224415/6224415 [21:00<00:00, 4937.09it/s]

Worker 0 finished processing samples.


Cached processed samples to C:\Users\shart\AppData\Local\pyhealth\pyhealth\Cache\c8f0e13c-fb2e-5216-8969-8e6afcc7338c\tasks\MVCLTrainingSleepEEG_2060b134-eac9-5c04-b238-25b00edd7ba5\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Total task samples: 6224415
Input schema: {'xt': 'tensor', 'xd': 'tensor', 'xf': 'tensor'}
Output schema: {'label': 'multiclass'}


In [6]:
# Factory functions and helpers required to setup the model.
import math
from pathlib import Path
from typing import Any, Dict, Dict, List, List, Mapping, Union

import torch
import torch.nn as nn

def augment_time(x: torch.Tensor, std: float = 0.1) -> torch.Tensor:
    """Time-domain jitter augmentation"""
    noise = torch.randn_like(x) * std
    return x + noise
    
def augment_freq(sample: torch.Tensor, pertub_ratio: float = 0.05) -> torch.Tensor:
    """Frequency-domain augmentation (remove and add frequencies)"""
    aug_1 = remove_frequency(sample, pertub_ratio)
    aug_2 = add_frequency(sample, pertub_ratio)
    return aug_1 + aug_2

def remove_frequency(x: torch.Tensor, pertub_ratio: float = 0.0) -> torch.Tensor:
    mask = torch.rand(x.shape, device=x.device) > pertub_ratio
    return x * mask

def add_frequency(x: torch.Tensor, pertub_ratio: float = 0.0) -> torch.Tensor:
    mask = torch.rand(x.shape, device=x.device) > (1 - pertub_ratio)
    max_amplitude = x.max()
    random_am = torch.rand(mask.shape, device=x.device) * (max_amplitude * 0.1)
    pertub_matrix = mask * random_am
    return x + pertub_matrix

class PositionalEncoding(nn.Module):
    def __init__(self, hidden_dim, dropout=0.1, max_len=1024):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, hidden_dim)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, hidden_dim, 2) * 
                             (-math.log(10000.0) / hidden_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # Shape: [1, max_len, hidden_dim]
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [7]:
# Now we set up the model and training loop.
from pyhealth.datasets.sample_dataset import SampleDataset
from pyhealth.datasets.splitter import split_by_sample
from pyhealth.datasets.utils import get_dataloader
from pyhealth.models.mvcl_model import MultiViewContrastiveModel
from torch.optim import Adam
from torch.utils.data import DataLoader
from pyhealth.trainer import Trainer

pretrain_ds, _, _ = split_by_sample(sample_dataset, [0.05, 0.05, 0.9])
assert type(pretrain_ds) == SampleDataset, "Expected a SampleDataset after splitting"
pretrain_loader = get_dataloader(pretrain_ds, batch_size=128, shuffle=True)


view_keys=["xt", "xf", "xd"]
model = MultiViewContrastiveModel(
    dataset=pretrain_ds,
    encoders=torch.nn.ModuleDict({
        k: nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=128, nhead=4, batch_first=True),
            num_layers=2
        ) for k in view_keys
    }),
    projectors = nn.ModuleDict({
        k: nn.Linear(1, 128) 
        for k in view_keys
    }),
    augmentations={"xt": augment_time, "xf": augment_freq, "xd": augment_time},
    pos_encoders=nn.ModuleDict({
        k: PositionalEncoding(hidden_dim=128, dropout=0.1) for k in view_keys
    }),
    hidden_dim=128,
    training_stage="pretrain",
    num_classes=3

)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

pretrainer = Trainer(
    model=model,
    device=str(device),
)
pretrainer.train(
    train_dataloader=pretrain_loader,
    epochs=2,
    optimizer_class=torch.optim.Adam,
    optimizer_params={
        "lr": 0.002,
        "weight_decay": 1e-5,
    },
)

MultiViewContrastiveModel(
  (encoders): ModuleDict(
    (xt): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=2048, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (xf): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=

Epoch 0 / 2:   0%|          | 0/2432 [00:00<?, ?it/s]

--- Train epoch-0, step-2432 ---
loss: 2.3579



Epoch 1 / 2:   0%|          | 0/2432 [00:00<?, ?it/s]

--- Train epoch-1, step-4864 ---
loss: 2.2124


In [8]:
# Save the pretraining model state for later use.
import os
import tempfile

tmp_path = tempfile.mkdtemp()
torch.save(model.state_dict(), os.path.join(tmp_path, "mvcl_pretrain.pth"))